# Dual-Student Knowledge Distillation (DSKD) - Baseline B0 Training on Kaggle GPU

**CS331 Anomaly Detection Project**  
Reference: *Dual-Student Knowledge Distillation Networks for Unsupervised Anomaly Detection* (arXiv:2402.00448v1)

This notebook trains the faithful baseline `B0-Clean` on Kaggle Tesla T4 GPU.

In [ ]:
# 1. Verify GPU availability
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:     ', torch.cuda.get_device_name(0))
    print(f'VRAM Total:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: No GPU detected. Please go to Kaggle Settings -> Accelerator -> GPU T4!')

In [ ]:
# 2. Clone repository or navigate to source directory
import os
from pathlib import Path

repo_dir = Path('/kaggle/working/cs331-knowledge-distillation')
if not repo_dir.exists():
    !git clone https://github.com/traineethanh/cs331-knowledge-distillation-lightweight-industrial-anomaly-detection.git {repo_dir}

%cd {repo_dir}
!git checkout chore/repository-reset
!pip install -q pyyaml scipy scikit-learn

In [ ]:
# 3. Verify MVTec AD Dataset
from collections import deque

def find_mvtec_root(input_root: Path, max_depth: int = 4) -> Path:
    queue = deque([(input_root, 0)])
    skip_names = {'train', 'test', 'ground_truth', '__pycache__'}
    while queue:
        current, depth = queue.popleft()
        if (current / 'carpet' / 'train').is_dir() or (current / 'bottle' / 'train').is_dir():
            return current
        if depth < max_depth:
            for child in sorted(current.iterdir()):
                if child.is_dir() and child.name not in skip_names:
                    queue.append((child, depth + 1))
    raise FileNotFoundError(
        f'Could not locate MVTec AD dataset root under {input_root}. '
        f'Top-level entries: {[p.name for p in input_root.iterdir()]}'
    )

mvtec_path = find_mvtec_root(Path('/kaggle/input'))
print(f'Using MVTec AD dataset directory: {mvtec_path}')

categories = sorted([p.name for p in mvtec_path.iterdir() if p.is_dir() and (p / 'train').is_dir()])
print(f'Available categories ({len(categories)}): {categories}')

## 4. Sequential Multi-Class Baseline Pipeline (All 15 Categories on GPU)

This cell sequentially runs training (200 epochs) and evaluation on GPU across all MVTec AD categories:
- **Sequential Processing:** Runs one class at a time, completely reclaiming GPU VRAM between classes.
- **Smart Skip/Resume (`--skip_existing`):** Automatically detects existing results (e.g. `carpet`) and skips re-training.
- **Incremental Live Summary:** Updates `summary_metrics.csv` and `summary_table.md` with group means (Texture Mean, Object Mean, Overall Mean) after each class.
- **Options for `--categories`:**
  - `all`: Full 15 categories benchmark
  - `core5`: 5 representative categories (`carpet`, `bottle`, `tile`, `screw`, `toothbrush`)
  - Or a specific list: `bottle tile screw`

In [ ]:
# Run sequential multi-class baseline on GPU
!python scripts/run_all_baseline.py \
    --config configs/baseline_b0.yaml \
    --data_root "{mvtec_path}" \
    --categories all \
    --skip_existing \
    --epochs 200 \
    --batch_size 16 \
    --device cuda \
    --output_dir results/baseline_b0

In [ ]:
# Display Benchmark Comparison Table vs Paper (Table 1 & Table 2)
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path

summary_md = Path('results/baseline_b0/summary_table.md')
summary_csv = Path('results/baseline_b0/summary_metrics.csv')

if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding='utf-8')))
elif summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df)
else:
    print('Summary results not found yet.')